# Claims Processing Pipeline — Demo

Two-agent pipeline: **ADR-1 (Intake)** → **ADR-4 (Triage)**

**ADR-1 runs in two internal steps:**
- **Step 1 — IDP Extraction** (Cell 2): format-specific parsers extract fields and assign confidence scores — no LLM involved
- **Step 2 — LLM Validation & Normalization** (Cell 3): Claude validates completeness, applies confidence thresholds, and produces the NormalizedClaimRecord

**How to use:** Change `CLAIM_FILE` in Cell 1 to any file in `mock-data/`.  
Supported formats: `.json` (portal), `.edi` (EDI 837P / 837I), `.txt` (CMS-1500 OCR).

Requires `ANTHROPIC_API_KEY` in your environment.

In [8]:
# ── Cell 1: Configuration ─────────────────────────────────────────────────────
import os, sys, json, re
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))
sys.path.insert(0, ".")

from preprocessors import preprocess
from prompts import INTAKE_SYSTEM_PROMPT, build_triage_prompt

# ── Change this to try different claims ──────────────────────────────────────
# CLAIM_FILE    = "../mock-data/portal-json/CLM-2026-1001201.json"  # fast path routine
# CLAIM_FILE  = "../mock-data/edi-837p/CLM-2026-1000001.edi"        # clinical path - minor surgery
# CLAIM_FILE  = "../mock-data/cms1500-ocr/CLM-2026-1001601.txt"     # human required - wrong cpt code
# CLAIM_FILE  = "../mock-data/edi-837i/CLM-2026-1001001.edi"        # clinical path - CPT 99285 (emergency dept E/M level 5)
# CLAIM_FILE  = "../mock-data/email/CLM-2026-1001901.eml"           # clinical path - prior auth required
# CLAIM_FILE  = "../mock-data/cms1500-ocr/CLM-2026-1001630.txt"     # human required - no first name
# CLAIM_FILE  = "../mock-data/portal-json/CLM-2026-1001220.json"    # clinical path - diagnostic imaging provision
# CLAIM_FILE  = "../mock-data/fhir-r4-json/CLM-2026-1001801.json"   # clinical path - high-complexity evaluation range
# CLAIM_FILE  = "../mock-data/llm-edge-cases/CLM-2026-9001.json"    # clinical path - 
# CLAIM_FILE  = "../mock-data/llm-edge-cases/CLM-2026-9002.txt"     # human_required - ICD and CPT incomplete codes
# CLAIM_FILE  = "../mock-data/llm-edge-cases/CLM-2026-9003.json"    # clinical path - Clinical implausibility: chemotherapy CPT + respiratory diagnosis
CLAIM_FILE  = "../mock-data/cms1500-ocr/CLM-2026-1001640.txt"

CODEBOOK_PATH = "../test-data/criteria-codebook-mock.json"
MODEL         = "claude-haiku-4-5-20251001"   # fast + cheap for the demo

print(f"Claim file:  {CLAIM_FILE}")
print(f"Codebook:    {CODEBOOK_PATH}")
print(f"Model:       {MODEL}")
print(f"API key set: {'ANTHROPIC_API_KEY' in os.environ}")

Claim file:  ../mock-data/cms1500-ocr/CLM-2026-1001640.txt
Codebook:    ../test-data/criteria-codebook-mock.json
Model:       claude-haiku-4-5-20251001
API key set: True


In [9]:
# ── Cell 2: ADR-1 — Step 1: IDP Extraction (simulated) ───────────────────────
preprocessed = preprocess(CLAIM_FILE)

print(f"Source format:  {preprocessed['source_format']}")
print(f"Claim ref:      {preprocessed['source_claim_ref']}")
print(f"Intake channel: {preprocessed['intake_channel']}")
print()
print("IDP extraction output (input to ADR-1 LLM step):")
print(json.dumps(preprocessed['extracted_fields'], indent=2))

Source format:  CMS1500_OCR_TEXT
Claim ref:      CLM-2026-1001640
Intake channel: CMS1500_OCR_TEXT

IDP extraction output (input to ADR-1 LLM step):
{
  "member_id": {
    "value": "U82434422",
    "confidence": 0.8400000000000001
  },
  "member_name_last": {
    "value": null,
    "confidence": 0.0
  },
  "member_name_first": {
    "value": null,
    "confidence": 0.0
  },
  "member_dob": {
    "value": null,
    "confidence": 0.0
  },
  "payer_id": {
    "value": null,
    "confidence": 0.0
  },
  "payer_name": {
    "value": "Cigna Open  ccess Plus - PPO",
    "confidence": 0.8
  },
  "plan_id": {
    "value": "GRP-3176",
    "confidence": 0.8
  },
  "date_of_service_start": {
    "value": "2026-03-28",
    "confidence": 0.8300000000000001
  },
  "date_of_service_end": {
    "value": "2026-04-07",
    "confidence": 0.8300000000000001
  },
  "place_of_service_code": {
    "value": "11",
    "confidence": 0.8600000000000001
  },
  "claim_type": {
    "value": "PROFESSIONAL",
    "conf

In [10]:
# ── Cell 3: ADR-1 — Step 2: LLM Validation & Normalization ───────────────────
import anthropic
from IPython.display import JSON

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env

print("Calling ADR-1 Step 2 (LLM Validation & Normalization)...")
response = client.messages.create(
    model=MODEL,
    max_tokens=4096,
    system=INTAKE_SYSTEM_PROMPT,
    messages=[{"role": "user", "content": json.dumps(preprocessed, indent=2)}]
)

raw_output = response.content[0].text

# Extract JSON robustly — the model sometimes adds a sentence before or after the JSON
json_match = re.search(r'\{.*\}', raw_output, re.DOTALL)
if not json_match:
    print("Raw model output (no JSON found):\n", raw_output)
    raise ValueError("ADR-1 response did not contain a JSON object.")
normalized_claim = json.loads(json_match.group())

status = normalized_claim.get("extraction_status")
low    = normalized_claim.get("low_confidence_fields", [])
print(f"\nExtraction status:    {status}")
if low:
    print(f"Low-confidence fields: {low}")
print(f"\nInput tokens:  {response.usage.input_tokens}")
print(f"Output tokens: {response.usage.output_tokens}")
print("\nNormalized Claim Record:")
display(JSON(normalized_claim))

Calling ADR-1 Step 2 (LLM Validation & Normalization)...

Extraction status:    HUMAN_REQUIRED
Low-confidence fields: ['member_name_last', 'member_name_first', 'cpt_codes', 'billing_provider_tax_id']

Input tokens:  11682
Output tokens: 1077

Normalized Claim Record:


<IPython.core.display.JSON object>

In [11]:
# ── Cell 4: Gate check ────────────────────────────────────────────────────────
status = normalized_claim.get("extraction_status")
low    = normalized_claim.get("low_confidence_fields", [])

if status == "HUMAN_REQUIRED":
    print("ROUTING: HUMAN_REVIEW (exception queue)")
    print(f"  Required field(s) below confidence threshold: {low}")
    print()
    print("ADR-4 triage is NOT called for HUMAN_REQUIRED claims.")
    print("This claim routes to the human exception queue for re-key.")

elif status == "PENDING_DUPLICATE":
    print("ROUTING: DUPLICATE HOLD")
    print("  Duplicate detected by CMS. Claim pended for processor review.")

elif status == "AUTO_COMPLETE":
    print("ROUTING: AUTO_COMPLETE")
    print("  All required fields present and above threshold.")
    print("  Proceeding to ADR-4 Clinical Content Triage (Cell 5).")

else:
    print(f"Unexpected extraction_status: {status}")

ROUTING: HUMAN_REVIEW (exception queue)
  Required field(s) below confidence threshold: ['member_name_last', 'member_name_first', 'cpt_codes', 'billing_provider_tax_id']

ADR-4 triage is NOT called for HUMAN_REQUIRED claims.
This claim routes to the human exception queue for re-key.


In [12]:
# ── Cell 5: ADR-4 — Clinical Content Triage Agent ────────────────────────────
import uuid

if normalized_claim.get("extraction_status") != "AUTO_COMPLETE":
    print("Skipped — claim is not AUTO_COMPLETE. See gate check output in Cell 4.")
else:
    # Simulate the CMS-assigned UUID (in production this comes from the CMS POST response)
    claim_uuid = str(uuid.uuid4())

    # Build minimal ADR-4 input (only the 9 fields the triage agent needs)
    triage_input = {
        "claim_id":            claim_uuid,
        "source_claim_ref":    normalized_claim.get("source_claim_ref"),
        "intake_channel":      normalized_claim.get("intake_channel"),
        "extraction_status":   normalized_claim.get("extraction_status"),
        "claim_type":          normalized_claim.get("claim_type"),
        "icd10_codes":         normalized_claim.get("icd10_codes"),
        "cpt_codes":           normalized_claim.get("cpt_codes"),
        "prior_auth_required": normalized_claim.get("prior_auth_required"),
        "prior_auth_number":   normalized_claim.get("prior_auth_number"),
    }

    triage_prompt = build_triage_prompt(CODEBOOK_PATH)

    print("Calling ADR-4 (Clinical Content Triage Agent)...")
    triage_response = client.messages.create(
        model=MODEL,
        max_tokens=4096,
        system=triage_prompt,
        messages=[{"role": "user", "content": json.dumps(triage_input, indent=2)}]
    )

    raw_triage = triage_response.content[0].text

    # Extract JSON robustly
    triage_match = re.search(r'\{.*\}', raw_triage, re.DOTALL)
    if not triage_match:
        print("Raw model output (no JSON found):\n", raw_triage)
        raise ValueError("ADR-4 response did not contain a JSON object.")
    triage_result = json.loads(triage_match.group())

    decision    = triage_result.get("routing_decision")
    confidence  = triage_result.get("confidence", 0)
    fallback    = triage_result.get("confidence_fallback", False)
    provisions  = triage_result.get("criteria_provisions_matched", [])
    indicators  = triage_result.get("clinical_indicators_detected", [])

    print(f"\nClaim ID (CMS UUID): {triage_result.get('claim_id')}")
    print(f"Source claim ref:    {triage_result.get('source_claim_ref')}")
    print(f"Routing decision:    {decision}")
    print(f"Confidence:          {confidence:.2f}")
    if fallback:
        print("  ** Confidence fallback applied (< 0.70 threshold) — overridden to CLINICAL_PATH")
    print(f"Provisions matched:  {provisions if provisions else '(none)'}")
    print(f"Indicators:          {indicators}")
    print(f"\nInput tokens:  {triage_response.usage.input_tokens}")
    print(f"Output tokens: {triage_response.usage.output_tokens}")
    print("\nFull triage output:")
    display(JSON(triage_result))

Skipped — claim is not AUTO_COMPLETE. See gate check output in Cell 4.
